# HEAT3: transfer entropy between companies on 25+ years of daily data

**Question:** does one S&P 500 stock's recent return tell us about another stock's return over the next
**1, 2, 5 or 10 trading days**, beyond what that stock's own past already tells us, positively or negatively,
and is it an edge after 2 bps?

**Data:** daily adjusted closes from **Yahoo Finance, 2000 onward**, downloaded once and kept on Drive.
Caveat for the write-up: Yahoo has no prices for companies that left the index, so the universe is today's
S&P 500 traced back (survivorship bias).

**How:**
1. **Transfer entropy between every pair of stocks**, every month, from the previous three years only:
   stock A's last day → stock B's next 1 / 2 / 5 / 10 days, after removing what B's own last day explains.
2. **Each stock's 5 peers = its 5 highest-TE sources, positive or negative.**
3. **TE becomes the models' feature:** the last 20 days of the stock and its peers, each peer's signed TE
   strength, and the TE-weighted peer signal go into BiLSTM, CNN-ResNet and Transformer models.
4. **Controls**, same model and budget: no peers, 5 random peers, 5 most *correlated* stocks. An **edge** must
   beat all three, earn more than 2 bps net on the test days (2019 onward; intervals resample 21-day blocks),
   hold in both halves, and survive false-discovery control over 12 cells.

The rule is registered in
[`configs/heat_v3.yaml`](https://github.com/Gaire-commits/Transfer-entropy-/blob/main/configs/heat_v3.yaml)
(2026-11-23), before any daily result. The 5-minute experiments (heat_v1, heat_v2) are still here: set the
environment variable `HEAT_CONFIG` before Setup to run one of them.

**The order:** pick the GPU → Setup → check run (inside the training years: train to 2010, score 2013–2014)
→ registered run → result → release days. Section 4 (tuning) is done: the default settings are used.

**Before anything else, pick the GPU:** Runtime → Change runtime type → A100, L4 or T4 GPU. Changing the
runtime type later resets the session, and then Setup must be run again. Don't use "Run all" once you reach
section 5: it scores the test days.

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.ismount("/content/drive"):
        raise SystemExit("Google Drive did not mount. Runtime -> Restart session, then run Setup again.")
    REPO = "/content/Transfer-entropy-"
    if not os.path.exists(REPO):
        !git clone -q https://github.com/Gaire-commits/Transfer-entropy-.git {REPO}
    else:
        !git -C {REPO} pull -q
    os.chdir(REPO)
    !pip install -q pyyaml yfinance
else:
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(REPO)
sys.path.insert(0, REPO)
for name in [m for m in sys.modules if m == "heat" or m.startswith("heat.")]:
    del sys.modules[name]  # re-running this cell after a pull uses the new code

import html as _html
import torch
from IPython.display import HTML, Markdown, display

import yaml
CONFIG = os.environ.get("HEAT_CONFIG", "configs/heat_v3.yaml")  # the registered rule: heat_v3 = Yahoo daily, 2000 on
CFG = yaml.safe_load(open(CONFIG))
EXP, SOURCE = CFG["experiment"], CFG["data"].get("source", "sip")
BASE = "/content/drive/MyDrive/entropy-gate-pipeline-data"
DATA = os.environ.get("HEAT_DATA_DIR", f"{BASE}/data/yahoo" if SOURCE == "yahoo" else f"{BASE}/data/bars_sip")
OUT = os.environ.get("HEAT_OUT_DIR", f"{BASE}/outputs/heat_v1")  # holds the heat_v1 tuning record
REG_OUT = os.path.join(os.path.dirname(OUT), EXP)  # this experiment's results
# Each experiment has its own panel file. The 5-minute SIP panel (~420 MB) stays on this machine's disk (a huge
# Drive upload can hold up the small result files); the daily Yahoo panel is small and is kept on Drive.
PANEL = os.environ.get("HEAT_PANEL", os.path.join(REG_OUT, "panel.npz") if SOURCE == "yahoo"
                       else f"/content/heat_panel_{EXP}.npz")


def sync_drive():
    """Never unmounts Drive (an earlier version did, and then files went to this machine's disk under a
    path that only looked like Drive). Checks Drive is really mounted and asks the OS to write files out;
    Google uploads them in the background. The download cells are the sure way to keep results."""
    if not IN_COLAB:
        return
    if not os.path.ismount("/content/drive"):
        print("WARNING: Google Drive is NOT mounted, so files are only on this machine and vanish with it.")
        print("Use the 'Keep a copy' download cells, or re-run Setup to mount Drive.")
        return
    os.sync()
    print("Drive: files written; Google uploads them in the background (the download cells are the sure copy).")


def download_results(out_dir, name):
    """Zip a run's results (reports, tables, the visual report; not the model weights) and download it
    to your computer. A copy that doesn't depend on Drive."""
    import zipfile
    out_dir = os.path.abspath(out_dir)
    if not os.path.isdir(out_dir):
        print("Nothing to download yet:", out_dir, "does not exist.")
        return
    path = f"/content/{name}.zip" if IN_COLAB else os.path.join(out_dir, f"{name}.zip")
    keep = (".md", ".csv", ".json", ".html", ".yaml")
    n = 0
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(out_dir):
            for f in files:
                if f.endswith(keep):
                    z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), out_dir))
                    n += 1
    print(f"{n} files in {path} ({os.path.getsize(path) / 1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files
        files.download(path)


def show_viz(out_dir, cfg_path="configs/heat_v1.yaml", height=1800):
    """Write <out_dir>/viz.html and show it here (it is also a file you can open full-screen)."""
    from heat import config
    from heat.viz import write_report
    cfg = config.load(cfg_path)
    meta = {"experiment": cfg["experiment"], "cost_bps": cfg["evaluation"]["cost_bps"],
            "level": cfg["evaluation"]["level"], "check": os.path.basename(out_dir) == "check"}
    path = write_report(out_dir, cfg["data"]["universe"], meta, PANEL)
    print("saved", path, "- download it from Drive to open it full-screen in any browser")
    page = open(path).read()
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # IPython suggests IFrame, which can't read a Drive file
        display(HTML('<iframe srcdoc="%s" width="100%%" height="%d" style="border:1px solid #8885;border-radius:8px"></iframe>'
                     % (_html.escape(page, quote=True), height)))


ARCHS = ("bilstm", "resnet1d", "transformer")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = {a: "cuda" for a in ARCHS}
else:
    print("=" * 78)
    print("No NVIDIA GPU. A100 / L4 / T4 GPU runtimes are the tested, fastest path (Runtime -> Change")
    print("runtime type). On a TPU runtime, testing whether the TPU can be used instead (about a minute)...")
    print("=" * 78)
    # In a separate process that exits when done: a TPU can be held by one process at a time, and a test
    # run inside this notebook would keep it from every script started afterwards.
    import json
    _lines = !{sys.executable} scripts/pick_device.py
    print("\n".join(l for l in _lines[:-1] if l.strip()))
    DEVICE = json.loads(_lines[-1])
    if set(DEVICE.values()) == {"cpu"}:
        print("Only the CPU works here: training will be slow. Switch to a GPU runtime if you can.")
DEVICE_ALL = max(set(DEVICE.values()), key=list(DEVICE.values()).count)  # for cells that run all three
print("training on:", DEVICE)
if SOURCE == "yahoo":
    print("data: Yahoo Finance daily bars, downloaded once to", DATA,
          "| already downloaded:", any(f.startswith("yahoo_daily") for f in os.listdir(DATA)) if os.path.isdir(DATA) else False)
else:
    print("SIP bars found:", os.path.isdir(os.path.join(DATA, "5Min")))
print("registered config:", CONFIG, "->", EXP, "| results:", REG_OUT)

## 2. Controls (optional: only after the code changed, 5–10 minutes)

Before real data: on synthetic stocks with a planted leader, the experiment must find the edge and
name the leader in the heat map; with nothing planted it must find nothing. Plus the no-look-ahead
checks, the tuning checks, the visual-report checks and the frozen-config check.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} -m pytest -q tests -W ignore

## 3. Check run on the real data (inside the training years)

Builds the panel once (downloads the data the first time), then a short run scored on years **inside the
registered training period**: for heat_v3, train to 2010, validate 2011–2012, score 2013–2014. It never touches
the test days, so it's safe to run as often as needed. It shows whether everything works and gives a feel for
the numbers; it is not the result.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --check --panel "$PANEL" --device "$DEVICE_ALL"
    report = os.path.join(REG_OUT, "check", "report.md")
    if os.path.exists(report):
        display(Markdown(open(report).read()))
    else:
        print("No report: the run above stopped with an error. Send a screenshot of its last lines.")
    sync_drive()  # make sure what the script wrote is really on Drive

### The check run, drawn

Transfer by hold, who leads whom (matrix and network), industries leading industries, and who moves
with whom. These are the check run's numbers, not the result.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    if os.path.exists(os.path.join(REG_OUT, "check", "cells.csv")) or os.path.exists(os.path.join(REG_OUT, "check", "cells_partial.csv")):
        show_viz(os.path.join(REG_OUT, "check"), CONFIG)
    else:
        print("No check results yet: run the check cell above first.")

**Keep a copy.** Downloads the check run's results to your computer as a zip (reports, tables, the
visual report). It doesn't depend on Drive, so do it even if Drive is slow. Send the zip, or a screenshot,
if Claude can't see the files on Drive.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    download_results(os.path.join(REG_OUT, "check"), EXP + "_check_results")

## 4. Tuning: DONE, nothing to run here

Tuning was done once, on heat_v1's validation days (2026-10-22): for all three models no setting beat the
defaults by more than the seed-to-seed noise, so the defaults were kept. heat_v2 and heat_v3 use them; their
registered configs say so. The tuning code is still in the repo (`scripts/tune.py`) for the record, but there
is nothing to run in this notebook.

## 5. The registered run: scores the test days

One cell trains and scores all three architectures. **Nothing is computed twice:** the peer and TE tables are
cached after the first run (the check run already made them), and every trained variant is saved the moment it
finishes. If the session drops, just run the cell again: finished variants are loaded, not retrained, and
it carries on from where it stopped.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    print("registered config:", CONFIG, "->", EXP)
    print("results go to:", REG_OUT)
    print("this scores the test days once (" + str(CFG["split"]["val_end"]) + " is the last validation day)")

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/run.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --panel "$PANEL" --device $DEVICE_ALL
    sync_drive()  # make sure what the script wrote is really on Drive

## 6. Result

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    report = os.path.join(REG_OUT, "report.md")
    if os.path.exists(report):
        display(Markdown(open(report).read()))
    else:
        print("No report yet: run the cells in section 5 first.")

### The result, drawn

Use the filters at the top of the report: architecture (or all three averaged), hold, and how many
links to show. Hover any square, arrow or dot for the numbers. Every chart has a table view.

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    if os.path.exists(os.path.join(REG_OUT, "cells.csv")) or os.path.exists(os.path.join(REG_OUT, "cells_partial.csv")):
        show_viz(REG_OUT, CONFIG)
    else:
        print("No results yet: run the cells in section 5 first.")

## 7. Macro release days (exploratory)

Does any transfer, or the edge, show up on **jobs report, CPI, GDP or FOMC days** rather than ordinary days?
This reuses the models already trained (their saved test predictions), so nothing is retrained; it is
exploratory, not a registered test, so it gives no verdict.

- Release dates are the **official schedules** (BLS jobs and CPI, BEA GDP, Federal Reserve FOMC), via FRED.
  Jobs, CPI and GDP come out at 8:30 am, before the open, so they are split by day. FOMC decisions come at
  2:00 pm, during trading, so FOMC is also split by whether a hold spans 2:00 pm.
- **One-time setup:** create a free FRED API key (fred.stlouisfed.org → My Account → API Keys) and add it
  as a Colab secret named `FRED_API_KEY` (key icon on the left, notebook access on). The calendar is saved
  the first time, so later runs don't need the key.

**7a. On the check run:**

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "OUT" not in globals():
    print("Not ready: run section 1 (Setup) first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/events.py --config "$CONFIG" --data-dir "$DATA" --out-dir "$REG_OUT" --panel "$PANEL" --check
    sync_drive()  # make sure what the script wrote is really on Drive

**7b. On the registered result (the test days), after section 5:**

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    !{sys.executable} scripts/events.py --panel "$PANEL" --config "$CONFIG" --data-dir "$DATA" --out-dir $REG_OUT
    sync_drive()  # make sure what the script wrote is really on Drive

**Keep a copy** of the registered result on your computer (independent of Drive).

In [ ]:
import os, sys
from IPython.display import Markdown, display
if "CONFIG" not in globals():
    print("Not ready: run the 'Choose the registered config' cell in section 5 first. (Colab resets everything when the runtime "
          "restarts, including after changing the runtime type.)")
else:
    download_results(REG_OUT, EXP + "_results")